In [1]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%                                       22.4%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
import subprocess
subprocess.Popen("ollama serve", stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, shell=True)
import time
time.sleep(3)

In [3]:
!ollama pull llama3.1:8b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠧ pulling manifest 
pulling 667b0c1932bc:   0% ▕                  ▏ 2.5 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   1% ▕                  ▏  27 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   2% ▕                  ▏  87 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   3% ▕                  ▏ 156 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   4% ▕                  ▏ 198 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   6% ▕█                 ▏ 277 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   7% ▕█                 ▏ 365 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   8% ▕█                 ▏ 405 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:  10% ▕█                 ▏ 481 MB/4.9 GB            

In [4]:
%%capture
!pip install dspy
! pip install litellm[proxy]

In [5]:
import dspy
lm = dspy.LM('ollama_chat/llama3.1:8b', api_base='http://localhost:11434', api_key='', temperature=100, cache=False)
dspy.configure(lm=lm)

In [6]:
import logging

for name in logging.root.manager.loggerDict:
    if 'litellm' in name.lower() or 'ollama' in name.lower():
        # print(name)
        logging.getLogger(name).setLevel(logging.ERROR)

for name in logging.root.manager.loggerDict:
    if 'dspy' in name.lower():
        # print(name)
        logging.getLogger(name).setLevel(logging.INFO)

In [7]:
import dspy
from typing import List

class ExperiencedStigmaPrompt(dspy.Signature):
    """
    Task: Generate new social media posts that reflect *Experienced (enacted) stigma*.

    Definition: Direct experiences of discrimination, exclusion, or negative treatment due to diabetes.

    Codebook Examples:
    - “I find a lot of people, they like to think of you as being the culprit…”
    - “The dietician was awful…she asked me if I exercise… ‘that’s not enough’…”
    - “His mother didn’t like me because I was diabetic… told him not to marry me…”
    """
    examples: List[str] = dspy.InputField(desc="Example posts showing enacted stigma")
    new_post: str = dspy.OutputField(desc="Generated social media post showing enacted stigma")

class PerceivedStigmaPrompt(dspy.Signature):
    """
    Task: Generate new social media posts that reflect *Perceived (felt) stigma*.

    Definition: Awareness or belief that others hold negative attitudes or stereotypes about people with diabetes.

    Codebook Example:
    - “There’s this message that diabetes is this terrible thing that terrible people get because they do terrible things.”
    """
    examples: List[str] = dspy.InputField(desc="Example posts showing perceived stigma")
    new_post: str = dspy.OutputField(desc="Generated social media post showing perceived stigma")

class AnticipatedStigmaPrompt(dspy.Signature):
    """
    Task: Generate new social media posts that reflect *Anticipated stigma*.

    Definition: The expectation or fear of experiencing diabetes stigma in the future.

    Codebook Example:
    - “When I started getting older… I used to hide it from any boyfriends…”
    """
    examples: List[str] = dspy.InputField(desc="Example posts showing anticipated stigma")
    new_post: str = dspy.OutputField(desc="Generated social media post showing anticipated stigma")

class InternalisedStigmaPrompt(dspy.Signature):
    """
    Task: Generate new social media posts that reflect *Internalised (self) stigma*.

    Definition: Acceptance and internalization of negative beliefs and stereotypes about oneself due to diabetes, leading to shame and self-blame.

    Codebook Examples:
    - “I felt guilty in the early days… it was my fault.”
    - “There is still somehow a feeling of stigma attached… you feel it’s your fault…”
    - “My brothers would never come and see me… they would say it was my own fault…”
    """
    examples: List[str] = dspy.InputField(desc="Example posts showing internalised stigma")
    new_post: str = dspy.OutputField(desc="Generated social media post showing internalised stigma")

class IntersectionalStigmaPrompt(dspy.Signature):
    """
    Task: Generate new social media posts that reflect *Intersectional stigma*.

    Definition: Diabetes stigma converging with other stigmatised conditions (e.g., obesity, schizophrenia) or characteristics (e.g., race, ethnicity).

    Codebook Example:
    - “Their appearance might be fat, they might look unhealthy and… that might be the main reason they’ve got a stigma rather than the diabetes.”
    """
    examples: List[str] = dspy.InputField(desc="Example posts showing intersectional stigma")
    new_post: str = dspy.OutputField(desc="Generated social media post showing intersectional stigma")


# Data filtering

In [ ]:
! gdown 
! gdown 

Downloading...
From: https://drive.google.com/uc?id=10hqgVjSkjgr19RtYmUzxASzBFAqAdKNx
To: /kaggle/working/train_data.json
100%|████████████████████████████████████████| 812k/812k [00:00<00:00, 78.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1mReHCi7WpGcb6043QgjkmJ4f9bIaDYk-
To: /kaggle/working/val_data.json
100%|████████████████████████████████████████| 124k/124k [00:00<00:00, 76.6MB/s]


In [9]:
import json
with open('/kaggle/working/train_data.json', 'r') as f:
  train_data = json.loads(f.read())
  f.close()

with open('/kaggle/working/val_data.json', 'r') as f:
  val_data = json.loads(f.read())
  f.close()

all_data = train_data + val_data

In [10]:
stigma_types = ['experienced stigma', 'perceived stigma', 'anticipated stigma', 'internalized stigma', 'intersectional stigma']

fewshot_samples = {x: [] for x in stigma_types}

for typ in stigma_types:
  for data_obj in all_data:
    if data_obj['label'] == 'yes stigma':
      if len(data_obj['types']) == 1 and typ in data_obj['types']:
        fewshot_samples[typ].append(data_obj)


In [11]:
for k in fewshot_samples:
  print(k, '=', len(fewshot_samples[k]))

experienced stigma = 24
perceived stigma = 7
anticipated stigma = 3
internalized stigma = 53
intersectional stigma = 1


In [12]:
type_generators = {'experienced stigma': dspy.Predict(ExperiencedStigmaPrompt),
                   'perceived stigma': dspy.Predict(PerceivedStigmaPrompt),
                   'anticipated stigma': dspy.Predict(AnticipatedStigmaPrompt),
                   'internalized stigma': dspy.Predict(InternalisedStigmaPrompt),
                   'intersectional stigma': dspy.Predict(IntersectionalStigmaPrompt)}

In [13]:
import random

key = 'anticipated stigma'

ans = type_generators[key](examples=random.sample(fewshot_samples[key], min(3, len(fewshot_samples[key])))).new_post

In [14]:
ans

'"I\'ve been trying to be more open about my diabetes, but I\'m worried about how others will react. Last week, I accidentally dropped my CGM in a coffee shop restroom, and it\'s been stuck there since. When someone asked if they could help me get it out, I panicked and quickly got the hint. Now I just sit with a bag over it trying to cover my blood work going down to try avoid a huge deal, like all I know this feeling but not know how others would react."'

In [15]:
import random
from tqdm import trange

generated_answers = {'experienced stigma': [],
                   'perceived stigma': [],
                   'anticipated stigma': [],
                   'internalized stigma': [],
                   'intersectional stigma': []}

for k in generated_answers:
  for _ in trange(10):
    ans = type_generators[k](examples=random.sample(fewshot_samples[k], min(3, len(fewshot_samples[k])))).new_post
    generated_answers[k].append(ans)


100%|██████████| 10/10 [01:03<00:00,  6.33s/it]


In [17]:
import json

with open('Llama3.1-samples.json', 'w') as f:
  f.write(json.dumps(generated_answers))
  f.close()